In [18]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver
import os

In [19]:
load_dotenv()

llm = ChatGroq(
    model="openai/gpt-oss-120b",    
    api_key=os.getenv("GROQ_API_KEY")
)

In [20]:
class JokeState(TypedDict):

    topic: str 
    joke: str 
    explanation: str

In [21]:
def generate_joke(state: JokeState):
    prompt = f'Generate a joke on a topic - {state['topic']}'
    response = llm.invoke(prompt).content

    return {'joke':response}

In [22]:
def generate_explanation(state: JokeState):
    prompt = f'Write an explanation for the joke - {state['joke']}'
    response = llm.invoke(prompt).content

    return {'explanation':response}

In [23]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [24]:
config1 = {"configurable":{"thread_id":1}}
workflow.invoke({"topic":'pizza'}, config=config1)

{'topic': 'pizza',
 'joke': 'Why did the pizza apply for a job?\n\nBecause it heard the company was looking for someone who could *deliver* under pressure and always *bring the cheese* to the table! 🍕😄',
 'explanation': '**Explanation of the joke**\n\n| Part of the joke | What it means literally | What the wordplay/pun refers to |\n|------------------|------------------------|---------------------------------|\n| **“Why did the pizza apply for a job?”** | Sets up a classic “knock‑knock”‑style question where the answer will be a pun. | The answer will hinge on the fact that a pizza is an inanimate food item, so “applying for a job” is already absurd and funny. |\n| **“Because it heard the company was looking for someone who could *deliver* under pressure…”** | • *Deliver* – to bring something to a destination. <br>• *Under pressure* – working while being stressed. | • **Deliver** is a common job‑skill phrase (“delivers results”). <br>• For a pizza, *delivering* is literally what a pizza

In [25]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza apply for a job?\n\nBecause it heard the company was looking for someone who could *deliver* under pressure and always *bring the cheese* to the table! 🍕😄', 'explanation': '**Explanation of the joke**\n\n| Part of the joke | What it means literally | What the wordplay/pun refers to |\n|------------------|------------------------|---------------------------------|\n| **“Why did the pizza apply for a job?”** | Sets up a classic “knock‑knock”‑style question where the answer will be a pun. | The answer will hinge on the fact that a pizza is an inanimate food item, so “applying for a job” is already absurd and funny. |\n| **“Because it heard the company was looking for someone who could *deliver* under pressure…”** | • *Deliver* – to bring something to a destination. <br>• *Under pressure* – working while being stressed. | • **Deliver** is a common job‑skill phrase (“delivers results”). <br>• For a pizza, *delivering* is li

## Time Travel

In [27]:
workflow.get_state({'configurable':{'thread_id':'1', 'checkpoint_id':'1f1a385d-4d7a-6f45-8001-922fc21af7ed'}})

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza apply for a job?\n\nBecause it heard the company was looking for someone who could *deliver* under pressure and always *bring the cheese* to the table! 🍕😄'}, next=('generate_explanation',), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f1a385d-4d7a-6f45-8001-922fc21af7ed'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-08-29T08:44:22.207835+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1a385d-47f0-6e88-8000-83e264f52edb'}}, tasks=(PregelTask(id='58b74dbd-8ea8-0948-17f3-9698075b5ef8', name='generate_explanation', path=('__pregel_pull', 'generate_explanation'), error=None, interrupts=(), state=None, result={'explanation': '**Explanation of the joke**\n\n| Part of the joke | What it means literally | What the wordplay/pun refers to |\n|------------------|------------------------|---------------------------------|\n| **“Why did t

In [28]:
workflow.invoke(None, {'configurable':{'thread_id':'1','checkpoint_id':'1f1a385d-4d7a-6f45-8001-922fc21af7ed'}})

{'topic': 'pizza',
 'joke': 'Why did the pizza apply for a job?\n\nBecause it heard the company was looking for someone who could *deliver* under pressure and always *bring the cheese* to the table! 🍕😄',
 'explanation': '**The Joke**\n\n> *Why did the pizza apply for a job?*  \n> *Because it heard the company was looking for someone who could **deliver** under pressure and always **bring the cheese** to the table!* 🍕😄  \n\n---\n\n## Why it’s funny – a step‑by‑step breakdown\n\n| Phrase in the punch‑line | What it means in everyday language | How it ties to a pizza (the “character”) |\n|--------------------------|-----------------------------------|------------------------------------------|\n| **“deliver”** | In a business context, “to deliver” means to meet expectations, finish a project, or provide a product/service on time and as promised. | A pizza’s primary job is literally to *be delivered* to a customer’s door. |\n| **“under pressure”** | “Under pressure” describes working in a 

In [29]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza apply for a job?\n\nBecause it heard the company was looking for someone who could *deliver* under pressure and always *bring the cheese* to the table! 🍕😄', 'explanation': '**The Joke**\n\n> *Why did the pizza apply for a job?*  \n> *Because it heard the company was looking for someone who could **deliver** under pressure and always **bring the cheese** to the table!* 🍕😄  \n\n---\n\n## Why it’s funny – a step‑by‑step breakdown\n\n| Phrase in the punch‑line | What it means in everyday language | How it ties to a pizza (the “character”) |\n|--------------------------|-----------------------------------|------------------------------------------|\n| **“deliver”** | In a business context, “to deliver” means to meet expectations, finish a project, or provide a product/service on time and as promised. | A pizza’s primary job is literally to *be delivered* to a customer’s door. |\n| **“under pressure”** | “Under pressure” des